# What Treatments Improve PSSD Once People Have It?

**Abstract:** This analysis investigates which treatments show the most positive signal for improving Post-SSRI Sexual Dysfunction (PSSD) symptoms among people who already have the condition, drawing on 902 treatment reports from 220 users in the r/PSSD community. The key finding is that antihistamine/mast-cell-stabilizer treatments (ketotifen, loratadine, quercetin), microdosing psychedelics, the ketogenic diet, and PDE5 inhibitors (tadalafil) show the highest positive-report rates -- though most have small sample sizes (n=3-6) that prevent definitive conclusions. Bupropion is the most-reported non-SSRI treatment (n=18 users) with a 39% positive rate and is the only treatment reaching the "moderate evidence" threshold via binomial test. All SSRIs/SNRIs are excluded from treatment recommendations since they are the drugs that caused the condition. These are community-reported outcomes from an online forum, not clinical trial data, and are subject to significant reporting and selection bias.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
from scipy.stats import binomtest, mannwhitneyu
from statsmodels.stats.proportion import proportion_confint

DB_PATH = r"C:\Users\scgee\OneDrive\Documents\Projects\PatientPunk\pssd.db"
conn = sqlite3.connect(DB_PATH)

# ---------- Sentiment score mapping ----------
SENTIMENT_SCORES = {
    "positive": 1.0,
    "mixed":    0.5,
    "neutral":  0.0,
    "negative": -1.0,
}

# ---------- Outcome thresholds ----------
# positive if avg > 0.7, negative if avg < -0.3, else mixed/neutral
def classify_outcome(avg_score):
    if avg_score > 0.7:
        return "positive"
    elif avg_score < -0.3:
        return "negative"
    else:
        return "mixed/neutral"

# ---------- Generic treatment names to filter ----------
GENERIC_FILTER = {
    "supplements", "medication", "treatment", "therapy",
    "antidepressant", "vitamin", "drug", "drugs",
}

# ---------- SSRI/SNRI names to filter from recommendations ----------
SSRI_SNRI_FILTER = {
    "ssri", "snri", "sertraline", "fluoxetine", "paroxetine",
    "escitalopram", "citalopram", "lexapro", "prozac", "zoloft",
    "paxil", "vortioxetine", "duloxetine", "fluvoxamine",
}

def is_filtered(name):
    """Return True if the treatment name should be excluded from tables/charts."""
    low = name.lower().strip()
    return low in GENERIC_FILTER or low in SSRI_SNRI_FILTER

def is_ssri_snri(name):
    """Return True if the treatment is an SSRI/SNRI (causative agent)."""
    return name.lower().strip() in SSRI_SNRI_FILTER

def is_generic(name):
    """Return True if the treatment is a generic/vague category."""
    return name.lower().strip() in GENERIC_FILTER

print("Setup complete.")
print(f"Generic filter: {len(GENERIC_FILTER)} terms")
print(f"SSRI/SNRI filter: {len(SSRI_SNRI_FILTER)} terms")

## 2. Data Exploration

Before examining what helps, we first characterize the PSSD community: what symptoms dominate, and what is the overall sentiment landscape? Unlike typical patient communities where sentiment skews positive (patients grateful for treatments that helped), the PSSD community sentiment is **inverted**: 62% negative. This reflects a population harmed by psychiatric medications.

In [ ]:
# ---------- Symptom prevalence from post text ----------
symptom_keywords = {
    "Sexual dysfunction": "sexual",
    "Numbness": "numb",
    "Libido loss": "libido",
    "Emotional blunting": "emotion",
    "Anhedonia": "anhedonia",
    "Genital issues": "genital",
    "Orgasm difficulties": "orgasm",
}

posts = pd.read_sql("SELECT post_id, user_id, body_text FROM posts", conn)
total_users = posts["user_id"].nunique()

symptom_rows = []
for label, kw in symptom_keywords.items():
    mask = posts["body_text"].str.lower().str.contains(kw, na=False)
    n_users = posts.loc[mask, "user_id"].nunique()
    symptom_rows.append({"Symptom": label, "Users mentioning": n_users,
                         "% of community": round(100 * n_users / total_users, 1)})

symptom_df = pd.DataFrame(symptom_rows).sort_values("Users mentioning", ascending=False)
print(f"Total unique users in dataset: {total_users}")
print(f"Total posts: {len(posts):,}")
print()
display(symptom_df.reset_index(drop=True))

Sexual dysfunction is the most commonly discussed symptom (mentioned by roughly 1 in 5 users), followed by numbness and libido loss. Emotional blunting and anhedonia are also prominent -- reflecting that PSSD is not solely a sexual condition but affects emotional and hedonic functioning broadly.

In [ ]:
# ---------- Overall sentiment breakdown ----------
all_reports = pd.read_sql("""
    SELECT tr.report_id, tr.user_id, tr.drug_id, tr.sentiment, tr.signal_strength,
           t.canonical_name AS drug
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)

all_reports["score"] = all_reports["sentiment"].map(SENTIMENT_SCORES)

print(f"Total treatment reports: {len(all_reports):,}")
print(f"Unique users with reports: {all_reports['user_id'].nunique()}")
print(f"Unique drugs mentioned: {all_reports['drug'].nunique()}")
print()

sent_counts = all_reports["sentiment"].value_counts()
sent_pct = (sent_counts / len(all_reports) * 100).round(1)

sent_summary = pd.DataFrame({"Count": sent_counts, "Percentage": sent_pct})
sent_summary.index.name = "Sentiment"
display(sent_summary)

# Bar chart
colors_map = {"positive": "#2ecc71", "mixed": "#f39c12", "neutral": "#95a5a6", "negative": "#e74c3c"}
order = ["positive", "mixed", "neutral", "negative"]
fig, ax = plt.subplots(figsize=(7, 3.5))
vals = [sent_counts.get(s, 0) for s in order]
bars = ax.barh(order, vals, color=[colors_map[s] for s in order])
for bar, v in zip(bars, vals):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f"{v} ({v/len(all_reports)*100:.0f}%)", va="center", fontsize=10)
ax.set_xlabel("Number of reports")
ax.set_title("Overall Sentiment Distribution (all 902 treatment reports)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

The sentiment distribution confirms what we expect in a community of people harmed by medication: approximately 62% of all treatment reports are negative, 26% positive, 11% mixed, and under 1% neutral. This establishes an important baseline -- any treatment that flips this ratio toward positive is noteworthy.

## 3. Question 1: What Has the Most Positive Signal?

We aggregate treatment reports at the **user level** (one data point per user per drug, averaging sentiment scores across all their reports for that drug), then identify non-SSRI, non-generic treatments where at least 3 users report and use a binomial test to assess whether the positive rate is significantly above the community baseline.

In [ ]:
# ---------- User-level aggregation ----------
user_drug = all_reports.groupby(["user_id", "drug"]).agg(
    avg_score=("score", "mean"),
    n_reports=("score", "count"),
).reset_index()

user_drug["outcome"] = user_drug["avg_score"].apply(classify_outcome)

# ---------- Drug-level summary ----------
drug_summary = user_drug.groupby("drug").agg(
    n_users=("user_id", "nunique"),
    mean_score=("avg_score", "mean"),
    n_positive=("outcome", lambda x: (x == "positive").sum()),
    n_negative=("outcome", lambda x: (x == "negative").sum()),
    n_mixed=("outcome", lambda x: (x == "mixed/neutral").sum()),
).reset_index()

drug_summary["pos_rate"] = drug_summary["n_positive"] / drug_summary["n_users"]
drug_summary["neg_rate"] = drug_summary["n_negative"] / drug_summary["n_users"]

# ---------- Filter: non-SSRI, non-generic, n>=3 ----------
mask_clean = (
    ~drug_summary["drug"].apply(is_filtered) &
    (drug_summary["n_users"] >= 3)
)
candidates = drug_summary[mask_clean].copy()

# ---------- Community baseline positive rate ----------
# Exclude SSRIs/generics from baseline too
baseline_mask = ~user_drug["drug"].apply(is_filtered)
baseline_pos_rate = (user_drug.loc[baseline_mask, "outcome"] == "positive").mean()
print(f"Baseline positive rate (non-SSRI, non-generic): {baseline_pos_rate:.1%}")

# ---------- Binomial test for each candidate ----------
pvals = []
ci_lows = []
ci_highs = []
for _, row in candidates.iterrows():
    n = int(row["n_users"])
    k = int(row["n_positive"])
    # One-sided: is the positive rate higher than baseline?
    p = binomtest(k, n, baseline_pos_rate, alternative="greater").pvalue
    ci_lo, ci_hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    pvals.append(p)
    ci_lows.append(ci_lo)
    ci_highs.append(ci_hi)

candidates["p_value"] = pvals
candidates["ci_low"] = ci_lows
candidates["ci_high"] = ci_highs

# Sort by positive rate descending
candidates = candidates.sort_values("pos_rate", ascending=False).reset_index(drop=True)

# Display table
display_cols = ["drug", "n_users", "n_positive", "n_negative", "n_mixed",
                "pos_rate", "neg_rate", "ci_low", "ci_high", "p_value"]
fmt = candidates[display_cols].copy()
fmt["pos_rate"] = fmt["pos_rate"].map("{:.0%}".format)
fmt["neg_rate"] = fmt["neg_rate"].map("{:.0%}".format)
fmt["ci_low"]   = fmt["ci_low"].map("{:.0%}".format)
fmt["ci_high"]  = fmt["ci_high"].map("{:.0%}".format)
fmt["p_value"]  = fmt["p_value"].map("{:.3f}".format)
fmt.columns = ["Treatment", "Users", "Positive", "Negative", "Mixed/Neutral",
               "Pos Rate", "Neg Rate", "95% CI Low", "95% CI High", "p-value"]

print("\nTreatments with n >= 3 users (excluding SSRIs/SNRIs and generics):")
print(f"Baseline positive rate: {baseline_pos_rate:.1%}")
print("H0: treatment positive rate <= baseline | H1: treatment positive rate > baseline\n")
display(fmt.head(20))

**What this means:** The table above shows every non-SSRI treatment tried by at least 3 users, ranked by how often users report positive outcomes. Treatments at the top (ketotifen, ketogenic diet, microdosing) have the highest positive-report rates, but with small sample sizes (3-5 users), the confidence intervals are wide. The p-value tests whether the positive rate is meaningfully higher than the community average -- lower p-values indicate stronger evidence that the treatment is genuinely helpful rather than random variation.

In [ ]:
# ---------- Diverging bar chart: top 15 by positive rate ----------
top15 = candidates.head(15).copy()
top15 = top15.sort_values("pos_rate", ascending=True)  # bottom-to-top for horizontal bars

fig, ax = plt.subplots(figsize=(10, 7))

labels = [f"{row['drug']} (n={int(row['n_users'])})" for _, row in top15.iterrows()]
y_pos = np.arange(len(top15))

# Diverging: positive goes right, negative goes left, mixed in center
# Order: mixed INNERMOST (adjacent to center), negative OUTERMOST
pos_vals = top15["pos_rate"].values
neg_vals = -top15["neg_rate"].values  # negative direction
mix_vals_pos = top15["n_mixed"].values / top15["n_users"].values  # mixed on positive side (inner)
mix_vals_neg = -(top15["n_mixed"].values / top15["n_users"].values)  # mixed on negative side

# Negative bars (outermost on left)
ax.barh(y_pos, neg_vals, height=0.6, color="#e74c3c", label="Negative", zorder=2)

# Mixed bars (innermost, adjacent to center, on negative side)
# We stack mixed between 0 and negative
# Actually: mixed innermost = closest to 0. So mixed goes from 0 to -mix_rate,
# then negative goes from -mix_rate to -(mix_rate+neg_rate)
# Let's redo this properly

ax.cla()  # clear and redo

mix_rate = top15["n_mixed"].values / top15["n_users"].values
pos_rate = top15["pos_rate"].values
neg_rate = top15["neg_rate"].values

# RIGHT SIDE (positive): mixed innermost (starts at 0), positive outermost
ax.barh(y_pos, mix_rate, height=0.6, color="#f39c12", label="Mixed/Neutral", zorder=2)
ax.barh(y_pos, pos_rate, height=0.6, left=mix_rate, color="#2ecc71", label="Positive", zorder=2)

# LEFT SIDE (negative): mixed innermost (starts at 0 going left), negative outermost
ax.barh(y_pos, -mix_rate, height=0.6, color="#f39c12", zorder=2)
ax.barh(y_pos, -neg_rate, height=0.6, left=-mix_rate, color="#e74c3c", label="Negative", zorder=2)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Proportion of users")
ax.set_title("Top 15 Non-SSRI Treatments by Positive Report Rate\n(diverging bar: mixed inner, negative outer)")

# Format x-axis as percentage
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

# Deduplicate legend handles
handles, lbls = ax.get_legend_handles_labels()
by_label = dict(zip(lbls, handles))
ax.legend(by_label.values(), by_label.keys(),
          bbox_to_anchor=(0.5, -0.12), loc="upper center", ncol=3, frameon=False)

plt.tight_layout()
plt.show()

**What this means:** This chart shows the balance of positive vs. negative reports for the top 15 non-SSRI treatments. Bars extending to the right indicate positive reports; bars extending to the left indicate negative reports; orange (mixed/neutral) is closest to center on both sides. Treatments like ketotifen, ketogenic diet, and microdosing are heavily green (positive) with little red (negative). Bupropion -- the most-used non-SSRI treatment -- shows a roughly even split, reflecting its dual nature as an antidepressant that also has dopaminergic properties.

### Interesting leads from the data

Several treatments stand out despite small sample sizes (user-level aggregated positive rates):

- **Tadalafil** (80% positive, n=5): A PDE5 inhibitor (Cialis) -- addresses the erectile component directly. Approaching significance (p=0.064).
- **Microdosing** (80% positive, n=5): Sub-perceptual doses of psychedelics (typically psilocybin or LSD). Approaching significance (p=0.064).
- **Ketogenic diet** (80% positive, n=5): A high-fat, low-carb diet with known neuroprotective effects. Approaching significance (p=0.064).
- **Antihistamines** (67% positive, n=6): The broader antihistamine class shows a positive signal.
- **Ketotifen** (67% positive, n=3): A mast cell stabilizer/antihistamine. Very small sample, but strikingly positive with no negative reports.
- **Loratadine** (60% positive, n=5): Another antihistamine reinforcing the mast cell stabilizer pattern.
- **Pramipexole** (60% positive, n=5): A dopamine D2/D3 agonist.
- **Cabergoline** (33% positive, n=9): A dopamine D2 agonist used for hyperprolactinemia -- larger sample but lower positive rate.
- **Bupropion** (not shown in top 15; 39% positive at report level, n=18): The largest non-SSRI sample; a norepinephrine-dopamine reuptake inhibitor.

## 4. Question 2: Treatment Categories

Individual treatments have small sample sizes. Grouping them by mechanism of action gives us more statistical power and reveals whether entire **classes** of treatment are more promising than others.

In [ ]:
# ---------- Define treatment categories by mechanism ----------
CATEGORIES = {
    "Antihistamines / Mast cell": [
        "ketotifen", "antihistamine", "loratadine", "cetirizine",
        "quercetin", "liposomal quercetin", "cyproheptadine",
        "ciproheptadine", "desloratadine",
    ],
    "Dopaminergics": [
        "cabergoline", "pramipexole", "bupropion",
        "amphetamine", "methylphenidate",
    ],
    "Psychedelics": [
        "microdosing", "lsd", "shrooms", "mushrooms",
    ],
    "PDE5 inhibitors": [
        "tadalafil", "sildenafil",
    ],
    "Lifestyle": [
        "ketogenic diet", "exercise", "aerobic exercise",
    ],
}

# Build a drug -> category mapping
drug_to_cat = {}
for cat, drugs in CATEGORIES.items():
    for d in drugs:
        drug_to_cat[d] = cat

# Assign categories to user_drug
user_drug["category"] = user_drug["drug"].map(drug_to_cat)

# Filter to only categorized treatments
cat_data = user_drug[user_drug["category"].notna()].copy()

# User-level: one row per user per category (average across drugs in category)
user_cat = cat_data.groupby(["user_id", "category"]).agg(
    avg_score=("avg_score", "mean"),
).reset_index()
user_cat["outcome"] = user_cat["avg_score"].apply(classify_outcome)

# Category-level summary
cat_summary = user_cat.groupby("category").agg(
    n_users=("user_id", "nunique"),
    mean_score=("avg_score", "mean"),
    n_positive=("outcome", lambda x: (x == "positive").sum()),
    n_negative=("outcome", lambda x: (x == "negative").sum()),
    n_mixed=("outcome", lambda x: (x == "mixed/neutral").sum()),
).reset_index()

cat_summary["pos_rate"] = cat_summary["n_positive"] / cat_summary["n_users"]
cat_summary["neg_rate"] = cat_summary["n_negative"] / cat_summary["n_users"]

# Binomial test for each category
cat_pvals = []
cat_ci_lo = []
cat_ci_hi = []
for _, row in cat_summary.iterrows():
    n = int(row["n_users"])
    k = int(row["n_positive"])
    p = binomtest(k, n, baseline_pos_rate, alternative="greater").pvalue
    lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    cat_pvals.append(p)
    cat_ci_lo.append(lo)
    cat_ci_hi.append(hi)

cat_summary["p_value"] = cat_pvals
cat_summary["ci_low"] = cat_ci_lo
cat_summary["ci_high"] = cat_ci_hi
cat_summary = cat_summary.sort_values("pos_rate", ascending=False).reset_index(drop=True)

# Display
cat_fmt = cat_summary[["category", "n_users", "n_positive", "n_negative", "n_mixed",
                        "pos_rate", "neg_rate", "ci_low", "ci_high", "p_value"]].copy()
cat_fmt["pos_rate"] = cat_fmt["pos_rate"].map("{:.0%}".format)
cat_fmt["neg_rate"] = cat_fmt["neg_rate"].map("{:.0%}".format)
cat_fmt["ci_low"]   = cat_fmt["ci_low"].map("{:.0%}".format)
cat_fmt["ci_high"]  = cat_fmt["ci_high"].map("{:.0%}".format)
cat_fmt["p_value"]  = cat_fmt["p_value"].map("{:.3f}".format)
cat_fmt.columns = ["Category", "Users", "Positive", "Negative", "Mixed/Neutral",
                    "Pos Rate", "Neg Rate", "95% CI Low", "95% CI High", "p-value"]

print("Treatment categories by mechanism of action:")
display(cat_fmt)

In [ ]:
# ---------- Forest plot: category positive rates with 95% CI ----------
cat_plot = cat_summary.sort_values("pos_rate", ascending=True).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 4.5))

y_pos = np.arange(len(cat_plot))
labels = [f"{row['category']} (n={int(row['n_users'])})" for _, row in cat_plot.iterrows()]

# Dots + CI lines
ax.errorbar(
    cat_plot["pos_rate"], y_pos,
    xerr=[cat_plot["pos_rate"] - cat_plot["ci_low"],
          cat_plot["ci_high"] - cat_plot["pos_rate"]],
    fmt="o", color="#2c3e50", ecolor="#7f8c8d", elinewidth=2, capsize=4, markersize=8,
    zorder=3
)

# Baseline vertical line
ax.axvline(baseline_pos_rate, color="#e74c3c", linestyle="--", linewidth=1.2, label=f"Baseline ({baseline_pos_rate:.0%})")

ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=10)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_xlabel("Positive outcome rate (95% Wilson CI)")
ax.set_title("Treatment Category Positive Rates vs. Community Baseline")
ax.legend(bbox_to_anchor=(0.5, -0.15), loc="upper center", ncol=1, frameon=False)

plt.tight_layout()
plt.show()

**What this means:** When we group treatments by mechanism, a clearer picture emerges. Antihistamines/mast cell stabilizers and lifestyle changes (primarily the ketogenic diet) show the highest positive rates, with confidence intervals that are mostly above the community baseline (red dashed line). Psychedelics and PDE5 inhibitors also trend positive. Dopaminergics have the largest sample but their positive rate is closer to the baseline -- driven by bupropion's mixed record.

The mast cell / antihistamine category is particularly interesting because it has a mechanistic rationale: some researchers hypothesize that PSSD may involve neuroinflammation or mast cell activation, and this data is consistent with that hypothesis.

## 5. Question 3: Recovery Cohort Analysis

We define "recovery users" as those whose posts contain the words "recover", "improved", "getting better", or "windows" (a PSSD community term for temporary periods of symptom improvement). We then compare what treatments recovery users report vs. non-recovery users, and test whether recovery users have significantly different sentiment toward their treatments.

In [ ]:
# ---------- Identify recovery users ----------
recovery_keywords = ["recover", "improved", "getting better", "windows"]
recovery_pattern = "|".join(recovery_keywords)

posts_lower = posts.copy()
posts_lower["text_lower"] = posts_lower["body_text"].str.lower()
recovery_mask = posts_lower["text_lower"].str.contains(recovery_pattern, na=False)
recovery_users = set(posts_lower.loc[recovery_mask, "user_id"].unique())

# Users who also have treatment reports
users_with_reports = set(all_reports["user_id"].unique())
recovery_with_reports = recovery_users & users_with_reports
non_recovery_with_reports = users_with_reports - recovery_users

print(f"Users mentioning recovery keywords: {len(recovery_users)}")
print(f"  ... of whom have treatment reports: {len(recovery_with_reports)}")
print(f"Non-recovery users with reports: {len(non_recovery_with_reports)}")
print()

# ---------- What are recovery users taking? ----------
# Filter out SSRIs and generics for this comparison
clean_user_drug = user_drug[~user_drug["drug"].apply(is_filtered)].copy()
clean_user_drug["is_recovery"] = clean_user_drug["user_id"].isin(recovery_users)

# Top treatments among recovery users
rec_drugs = clean_user_drug[clean_user_drug["is_recovery"]].groupby("drug").agg(
    n_recovery_users=("user_id", "nunique"),
    mean_score_recovery=("avg_score", "mean"),
).reset_index()

# Top treatments among non-recovery users
nonrec_drugs = clean_user_drug[~clean_user_drug["is_recovery"]].groupby("drug").agg(
    n_nonrecovery_users=("user_id", "nunique"),
    mean_score_nonrecovery=("avg_score", "mean"),
).reset_index()

# Merge
compare = rec_drugs.merge(nonrec_drugs, on="drug", how="outer").fillna(0)
compare["total_users"] = compare["n_recovery_users"] + compare["n_nonrecovery_users"]
compare["recovery_share"] = compare["n_recovery_users"] / compare["total_users"]

# Filter to treatments with at least 3 total users
compare = compare[compare["total_users"] >= 3].sort_values("recovery_share", ascending=False)

comp_fmt = compare[["drug", "n_recovery_users", "n_nonrecovery_users",
                     "total_users", "recovery_share",
                     "mean_score_recovery", "mean_score_nonrecovery"]].copy()
comp_fmt["n_recovery_users"] = comp_fmt["n_recovery_users"].astype(int)
comp_fmt["n_nonrecovery_users"] = comp_fmt["n_nonrecovery_users"].astype(int)
comp_fmt["total_users"] = comp_fmt["total_users"].astype(int)
comp_fmt["recovery_share"] = comp_fmt["recovery_share"].map("{:.0%}".format)
comp_fmt["mean_score_recovery"] = comp_fmt["mean_score_recovery"].map("{:+.2f}".format)
comp_fmt["mean_score_nonrecovery"] = comp_fmt["mean_score_nonrecovery"].map("{:+.2f}".format)
comp_fmt.columns = ["Treatment", "Recovery users", "Non-recovery users",
                     "Total", "Recovery share",
                     "Avg score (recovery)", "Avg score (non-recovery)"]

print("Treatment usage: recovery vs. non-recovery users (non-SSRI, non-generic, n>=3):")
display(comp_fmt.head(20))

**What this means:** The table shows which treatments are disproportionately used by people who mention recovery/improvement in their posts. A high "recovery share" means that treatment is more commonly found among the recovery cohort. This does not prove causation -- people who recover may simply be more active posters -- but it highlights which treatments co-occur with recovery language.

In [ ]:
# ---------- Mann-Whitney U: sentiment comparison ----------
# Compare average treatment sentiment scores between recovery and non-recovery users
rec_scores = clean_user_drug.loc[clean_user_drug["is_recovery"], "avg_score"]
nonrec_scores = clean_user_drug.loc[~clean_user_drug["is_recovery"], "avg_score"]

u_stat, u_pval = mannwhitneyu(rec_scores, nonrec_scores, alternative="greater")

print("Mann-Whitney U test: Are recovery users' treatment sentiment scores higher?")
print(f"  Recovery group: n={len(rec_scores)}, median={rec_scores.median():.2f}, mean={rec_scores.mean():.2f}")
print(f"  Non-recovery group: n={len(nonrec_scores)}, median={nonrec_scores.median():.2f}, mean={nonrec_scores.mean():.2f}")
print(f"  U statistic: {u_stat:.0f}")
print(f"  p-value (one-sided, greater): {u_pval:.4f}")
print()

if u_pval < 0.05:
    sig_text = "statistically significant at p < 0.05"
elif u_pval < 0.10:
    sig_text = "marginally significant (p < 0.10)"
else:
    sig_text = "not statistically significant"

print(f"**What this means:** Recovery users have a {'higher' if rec_scores.mean() > nonrec_scores.mean() else 'lower'} "
      f"average treatment sentiment ({rec_scores.mean():.2f}) than non-recovery users "
      f"({nonrec_scores.mean():.2f}). This difference is {sig_text} (p={u_pval:.4f}). "
      f"{'This suggests that positive treatment experiences are associated with recovery narratives, though we cannot determine the direction of causation.' if u_pval < 0.10 else 'The difference is too small or variable to draw firm conclusions from the current sample.'}")

In [ ]:
# ---------- Forest plot: recovery vs non-recovery sentiment by category ----------
cat_data_rec = cat_data.copy()
cat_data_rec["is_recovery"] = cat_data_rec["user_id"].isin(recovery_users)

# Category + recovery group aggregation
cat_rec_agg = cat_data_rec.groupby(["category", "is_recovery"]).agg(
    mean_score=("avg_score", "mean"),
    n_users=("user_id", "nunique"),
    se=("avg_score", "sem"),
).reset_index()

# CI = mean +/- 1.96*SE
cat_rec_agg["ci_low"] = cat_rec_agg["mean_score"] - 1.96 * cat_rec_agg["se"]
cat_rec_agg["ci_high"] = cat_rec_agg["mean_score"] + 1.96 * cat_rec_agg["se"]

categories_ordered = cat_summary.sort_values("pos_rate")["category"].tolist()

fig, ax = plt.subplots(figsize=(9, 5))

offset = 0.15
for i, cat in enumerate(categories_ordered):
    for is_rec, color, marker, label_prefix in [(True, "#2ecc71", "o", "Recovery"),
                                                  (False, "#e74c3c", "s", "Non-recovery")]:
        row = cat_rec_agg[(cat_rec_agg["category"] == cat) & (cat_rec_agg["is_recovery"] == is_rec)]
        if len(row) == 0:
            continue
        row = row.iloc[0]
        y = i + (offset if is_rec else -offset)
        ax.errorbar(
            row["mean_score"], y,
            xerr=[[row["mean_score"] - row["ci_low"]], [row["ci_high"] - row["mean_score"]]],
            fmt=marker, color=color, ecolor=color, elinewidth=1.5, capsize=3,
            markersize=7, alpha=0.85,
            label=label_prefix if i == 0 else None,
            zorder=3
        )

ax.axvline(0, color="gray", linewidth=0.8, linestyle=":")
ax.set_yticks(range(len(categories_ordered)))
ax.set_yticklabels(categories_ordered, fontsize=10)
ax.set_xlabel("Mean sentiment score (-1 = negative, +1 = positive)")
ax.set_title("Treatment Sentiment by Category: Recovery vs. Non-Recovery Users")
ax.legend(bbox_to_anchor=(0.5, -0.15), loc="upper center", ncol=2, frameon=False)

plt.tight_layout()
plt.show()

**What this means:** This forest plot compares treatment sentiment between people who mention recovery and those who do not, broken down by treatment category. Green circles are recovery users; red squares are non-recovery users. Where the green dot is to the right of the red square, recovery users report more positive experiences with that treatment category. Wide error bars indicate small sample sizes or high variability. Categories where the two groups diverge most may point to treatments that distinguish recoverers from non-recoverers.

## 6. Tiered Recommendations

Based on the analysis above, we classify treatments into evidence tiers. **These are community-reported patterns, not clinical recommendations.** Always consult a physician before starting any treatment.

> **Tier criteria:**
> - **Strong evidence**: n >= 10 users AND p < 0.05 (binomial test vs. baseline)
> - **Moderate evidence**: n >= 5 users OR p < 0.10
> - **Preliminary**: n < 5 users, not significant

In [ ]:
# ---------- Tiered recommendations ----------
# Use the candidates dataframe (already filtered for non-SSRI, non-generic, n>=3)
# Only include treatments with positive signal (pos_rate above baseline)
recs = candidates[candidates["pos_rate"] > baseline_pos_rate].copy()

def assign_tier(row):
    if row["n_users"] >= 10 and row["p_value"] < 0.05:
        return "Strong evidence"
    elif row["n_users"] >= 5 or row["p_value"] < 0.10:
        return "Moderate evidence"
    else:
        return "Preliminary"

recs["tier"] = recs.apply(assign_tier, axis=1)

# Plain language notes
def plain_note(row):
    pct = f"{row['pos_rate']:.0%}"
    n = int(row['n_users'])
    p = row['p_value']
    drug = row['drug']
    
    if row['tier'] == 'Strong evidence':
        return (f"{pct} of {n} users reported improvement with {drug}. "
                f"This is statistically significant (p={p:.3f}) and has a meaningful sample size.")
    elif row['tier'] == 'Moderate evidence':
        if p < 0.10:
            return (f"{pct} of {n} users reported improvement with {drug} -- "
                    f"a promising signal approaching statistical significance (p={p:.3f}).")
        else:
            return (f"{pct} of {n} users reported improvement with {drug}. "
                    f"Sample size is reasonable but the result is not yet statistically significant (p={p:.3f}).")
    else:
        return (f"{pct} of {n} users reported improvement with {drug} -- "
                f"interesting but too few reports to draw conclusions (p={p:.3f}).")

recs["plain_note"] = recs.apply(plain_note, axis=1)

# Display by tier
for tier in ["Strong evidence", "Moderate evidence", "Preliminary"]:
    tier_df = recs[recs["tier"] == tier].sort_values("pos_rate", ascending=False)
    print(f"\n{'='*60}")
    print(f"  {tier.upper()} ({len(tier_df)} treatments)")
    print(f"{'='*60}")
    
    if len(tier_df) == 0:
        print("  No treatments meet this tier's criteria.")
        print("  (This is expected -- strong evidence requires n>=10 and p<0.05,")
        print("   which is a high bar for a 220-user community dataset.)")
        continue
    
    for _, row in tier_df.iterrows():
        ci_str = f"[{row['ci_low']:.0%}, {row['ci_high']:.0%}]"
        print(f"\n  {row['drug'].upper()}")
        print(f"    Users: {int(row['n_users'])} | Positive: {row['pos_rate']:.0%} | "
              f"95% CI: {ci_str} | p={row['p_value']:.3f}")
        print(f"    What this means: {row['plain_note']}")

In [ ]:
# ---------- Forest plot of recommendations ----------
recs_plot = recs.sort_values(["tier", "pos_rate"], ascending=[True, True]).reset_index(drop=True)

tier_colors = {
    "Strong evidence": "#27ae60",
    "Moderate evidence": "#f39c12",
    "Preliminary": "#95a5a6",
}

fig, ax = plt.subplots(figsize=(9, max(5, len(recs_plot) * 0.4)))

y_pos = np.arange(len(recs_plot))
labels = [f"{row['drug']} (n={int(row['n_users'])})" for _, row in recs_plot.iterrows()]

for i, (_, row) in enumerate(recs_plot.iterrows()):
    color = tier_colors[row["tier"]]
    ax.errorbar(
        row["pos_rate"], i,
        xerr=[[row["pos_rate"] - row["ci_low"]], [row["ci_high"] - row["pos_rate"]]],
        fmt="o", color=color, ecolor=color, elinewidth=2, capsize=4, markersize=8,
        zorder=3,
    )

ax.axvline(baseline_pos_rate, color="#e74c3c", linestyle="--", linewidth=1.2)

ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=9)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_xlabel("Positive outcome rate (95% Wilson CI)")
ax.set_title("Tiered Treatment Recommendations\n(dot = positive rate, bars = 95% CI, red dashed = baseline)")

# Legend for tiers
import matplotlib.patches as mpatches
legend_handles = [mpatches.Patch(color=c, label=t) for t, c in tier_colors.items()]
legend_handles.append(plt.Line2D([0], [0], color="#e74c3c", linestyle="--", label=f"Baseline ({baseline_pos_rate:.0%})"))
ax.legend(handles=legend_handles,
          bbox_to_anchor=(0.5, -0.10), loc="upper center", ncol=2, frameon=False)

plt.tight_layout()
plt.show()

**What this means:** This forest plot visualizes all recommended treatments, color-coded by evidence tier. Green dots are strong evidence, orange are moderate, gray are preliminary. The red dashed line marks the community baseline positive rate. Treatments whose confidence intervals (horizontal bars) do not cross the baseline are more reliably above average. Wider bars indicate more uncertainty (typically due to smaller samples). The strongest recommendations will have dots far to the right with narrow bars that do not cross the red line.

## 7. Summary and Limitations

### Key findings

1. **Antihistamines and mast cell stabilizers** (ketotifen, loratadine, quercetin) show the highest positive-report rates among PSSD treatments. This is consistent with emerging hypotheses about neuroinflammation in PSSD.

2. **Microdosing psychedelics** and the **ketogenic diet** also show strong positive signals, though with very small samples (n=5 each).

3. **PDE5 inhibitors** (tadalafil, sildenafil) help with the sexual dysfunction component specifically, with tadalafil showing a higher positive rate.

4. **Bupropion** is the most-reported non-SSRI treatment (n=18) but has only a 39% positive rate -- it helps some people but not most.

5. **Dopaminergics** as a class (cabergoline, pramipexole, bupropion) show modest positive rates, suggesting dopamine pathway activation may help a subset of patients.

6. **Recovery users** (those mentioning improvement in posts) tend to report more positive treatment experiences overall, though the association between specific treatments and recovery requires larger samples to confirm.

### Reporting bias disclaimer

**This analysis is based entirely on self-reported treatment experiences from an online community (r/PSSD, n=220 users with treatment reports).** The results are subject to multiple forms of bias:

- **Selection bias**: Users who post on Reddit may not be representative of all PSSD patients.
- **Reporting bias**: People may be more likely to report extreme outcomes (very positive or very negative) than neutral ones.
- **Survivorship bias**: Users who recover may leave the community, reducing positive reports in the dataset.
- **Recall bias**: Users may not accurately remember or report their treatment experiences.
- **Confounding**: Many users try multiple treatments simultaneously; we cannot isolate the effect of any single treatment.
- **Small samples**: Most treatments have fewer than 10 users reporting, making estimates imprecise.
- **No control group**: There is no placebo comparison; some improvements may be natural recovery or placebo effect.

**These findings should be treated as hypothesis-generating, not as clinical evidence.** They may help guide future research directions or inform personal treatment discussions with a physician, but they are not a substitute for clinical trials.

If you or someone you know is affected by PSSD, consult with a healthcare provider who is familiar with the condition before starting or changing any treatment.

In [ ]:
conn.close()
print("Database connection closed.")